# `index = row * n_cols + col`, all the way down to a physical DRAM cell

The row-major flat-index formula shows up everywhere in this repo — it's the
tensor-stride formula in
[`dgs/torch/cuda_pointer_arithmetic.py`](../dgs/torch/cuda_pointer_arithmetic.py),
it's how a CUDA kernel finds its thread's element, it's how NumPy lays out a
2D array in memory. This notebook asks what's underneath the arithmetic: on
an actual memory chip, nothing "computes row*n_cols+col" — a **row decoder**
and a **column decoder**, each a bank of literal Boolean AND gates
(exactly [`dgs/boolean_algebra.py`](../dgs/boolean_algebra.py)'s minterm
construction), select one wordline and one bitline, and the cell where they
cross is the one that gets read.

[`dgs/memory_address_decoder.py`](../dgs/memory_address_decoder.py) builds
that gate-level decoder from scratch, proves it's exhaustively equivalent to
the arithmetic formula, and wires the result into an actual physical DRAM
cell model from [`dgs/memory_circuits.py`](../dgs/memory_circuits.py) — so
"decoding the wrong address" here means literally reading a different cell's
real, independently-decaying voltage, not an abstract error code.


In [1]:
import sys, pathlib
import numpy as np

REPO = pathlib.Path(r"D:/Summer2026/Dispersion-Assisted-GS-Phase-Recovery")
sys.path.insert(0, str(REPO))
from dgs import memory_address_decoder as mad
from dgs import memory_circuits as mc
from dgs.torch import cuda_pointer_arithmetic as cpa

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'}  —  {label}")


## 1. A decoder output line IS a Boolean minterm

An $n$-to-$2^n$ decoder's $j$-th output is the AND of every address bit (or
its complement) matching $j$'s binary pattern — the textbook minterm
definition, built here as literal gate logic, not a lookup table.


In [2]:
print("3-bit address 011 (=3):")
for j in range(8):
    print(f"  minterm {j} ({mad.address_to_bits(j, 3)}): {mad.minterm((0, 1, 1), j)}")

check("Exactly one minterm fires for address (0,1,1)",
      sum(mad.minterm((0, 1, 1), j) for j in range(8)) == 1)
check("The firing minterm is j=3 (binary 011)", mad.minterm((0, 1, 1), 3) == 1)


3-bit address 011 (=3):
  minterm 0 ((0, 0, 0)): 0
  minterm 1 ((0, 0, 1)): 0
  minterm 2 ((0, 1, 0)): 0
  minterm 3 ((0, 1, 1)): 1
  minterm 4 ((1, 0, 0)): 0
  minterm 5 ((1, 0, 1)): 0
  minterm 6 ((1, 1, 0)): 0
  minterm 7 ((1, 1, 1)): 0
PASS  —  Exactly one minterm fires for address (0,1,1)
PASS  —  The firing minterm is j=3 (binary 011)


## 2. The decoder is one-hot for EVERY address (exhaustive, not sampled)

Because the address space is small enough to enumerate completely, this
checks all $2^n$ addresses for $n=1..8$ — a real exhaustive proof, not a
handful of spot checks.


In [3]:
for n_bits in range(1, 9):
    ok = mad.decoder_is_onehot(n_bits)
    print(f"n_bits={n_bits}  ({2**n_bits:4d} addresses):  one-hot for all = {ok}")
    check(f"{n_bits}-bit decoder is one-hot for every address", ok)


n_bits=1  (   2 addresses):  one-hot for all = True
PASS  —  1-bit decoder is one-hot for every address
n_bits=2  (   4 addresses):  one-hot for all = True
PASS  —  2-bit decoder is one-hot for every address
n_bits=3  (   8 addresses):  one-hot for all = True
PASS  —  3-bit decoder is one-hot for every address
n_bits=4  (  16 addresses):  one-hot for all = True
PASS  —  4-bit decoder is one-hot for every address
n_bits=5  (  32 addresses):  one-hot for all = True
PASS  —  5-bit decoder is one-hot for every address
n_bits=6  (  64 addresses):  one-hot for all = True
PASS  —  6-bit decoder is one-hot for every address
n_bits=7  ( 128 addresses):  one-hot for all = True
PASS  —  7-bit decoder is one-hot for every address
n_bits=8  ( 256 addresses):  one-hot for all = True
PASS  —  8-bit decoder is one-hot for every address


## 3. Gate-level decoder vs. arithmetic flat index — cross-checked exhaustively

`decode_cell(row, col, ...)` builds the 2D AND-of-(row line, column line)
select grid. `flat_address(row, col, n_cols)` reuses
`cuda_pointer_arithmetic.flat_index_from_multi_index` verbatim — the exact
function already trusted by `test_cuda_pointer_arithmetic.py` for tensor
strides. Every $(row, col)$ pair, across several row/column bit widths, must
agree on which single linear position gets selected.


In [4]:
for n_row_bits, n_col_bits in [(1, 1), (2, 2), (2, 3), (3, 2), (3, 3)]:
    ok = mad.decoder_matches_flat_index(n_row_bits, n_col_bits)
    n_rows, n_cols = 2**n_row_bits, 2**n_col_bits
    print(f"{n_row_bits}-bit row x {n_col_bits}-bit col  ({n_rows}x{n_cols} grid, "
          f"{n_rows*n_cols} addresses):  matches = {ok}")
    check(f"Gate decoder matches flat index for {n_row_bits}x{n_col_bits}-bit addressing", ok)

# and directly, via the same stride-based formula cuda_pointer_arithmetic uses elsewhere
n_cols = 4
for row, col, expected in [(0, 0, 0), (1, 2, 6), (3, 3, 15)]:
    got = mad.flat_address(row, col, n_cols)
    via_stride_formula = cpa.flat_index_from_multi_index((row, col), (n_cols, 1))
    check(f"flat_address({row},{col}) == {expected} == stride formula",
          got == expected == via_stride_formula)


1-bit row x 1-bit col  (2x2 grid, 4 addresses):  matches = True
PASS  —  Gate decoder matches flat index for 1x1-bit addressing
2-bit row x 2-bit col  (4x4 grid, 16 addresses):  matches = True
PASS  —  Gate decoder matches flat index for 2x2-bit addressing
2-bit row x 3-bit col  (4x8 grid, 32 addresses):  matches = True
PASS  —  Gate decoder matches flat index for 2x3-bit addressing
3-bit row x 2-bit col  (8x4 grid, 32 addresses):  matches = True
PASS  —  Gate decoder matches flat index for 3x2-bit addressing
3-bit row x 3-bit col  (8x8 grid, 64 addresses):  matches = True
PASS  —  Gate decoder matches flat index for 3x3-bit addressing
PASS  —  flat_address(0,0) == 0 == stride formula
PASS  —  flat_address(1,2) == 6 == stride formula
PASS  —  flat_address(3,3) == 15 == stride formula


## 4. From gate to voltage: decoding the wrong address reads a different real cell

Build a small memory array where every cell holds its own, independently
decaying DRAM voltage (`dgs.memory_circuits.dram_cell_decay`, each cell read
at a different moment). Request one specific `(row, col)` through the
Boolean decoder, then show what a decoder bug landing on a different address
would *actually* read — a genuinely different physical voltage, not a
symbolic mismatch.


In [5]:
n_row_bits, n_col_bits = 2, 2
n_rows, n_cols = 2**n_row_bits, 2**n_col_bits

rng = np.random.default_rng(0)
V0 = 3.3
access_time_s = rng.uniform(1e-3, 8e-3, size=(n_rows, n_cols))
R_leak, C_cell = 3e12, 30e-15
cell_voltages = mc.dram_cell_decay(V0, access_time_s, R_leak, C_cell)

print("Cell voltage grid (V):")
print(np.round(cell_voltages, 4))

target_row, target_col = 2, 1
correct = mad.read_cell_voltage(cell_voltages, target_row, target_col, n_row_bits, n_col_bits)

wrong_row, wrong_col = 0, 3
miswired = cell_voltages[wrong_row, wrong_col]

print(f"\nRequested (row={target_row}, col={target_col}): {correct:.4f} V")
print(f"A decoder bug landing on (row={wrong_row}, col={wrong_col}) instead: {miswired:.4f} V")

check("read_cell_voltage retrieves exactly the requested cell",
      correct == cell_voltages[target_row, target_col])
check("A different address reads a genuinely different voltage",
      correct != miswired)


Cell voltage grid (V):
[[3.1058 3.1958 3.2532 3.2593]
 [3.0635 3.0399 3.1131 3.0835]
 [3.1284 3.0346 3.0629 3.2628]
 [3.053  3.255  3.0835 3.2193]]

Requested (row=2, col=1): 3.0346 V
A decoder bug landing on (row=0, col=3) instead: 3.2593 V
PASS  —  read_cell_voltage retrieves exactly the requested cell
PASS  —  A different address reads a genuinely different voltage


## 5. GHz clock budget: how many decode cycles fit before a refresh is forced

Ties the Boolean decoder's per-cycle operation to the physical retention
budget from `dgs.memory_circuits.dram_refresh_interval` — "GHz solid-state
memory" isn't just a clock speed number, it's how many address-decode
cycles you get to spend before the physics forces a refresh.


In [6]:
V_threshold = 1.5
for clock_ghz in (1.6, 3.2, 6.4):
    n_access = mad.accesses_per_refresh_interval(clock_ghz * 1e9, V0, V_threshold, R_leak, C_cell)
    print(f"clock = {clock_ghz:4.1f} GHz  ->  {n_access:>15,.0f} address-decode cycles per refresh interval")

budget_low = mad.accesses_per_refresh_interval(1.6e9, V0, V_threshold, R_leak, C_cell)
budget_high = mad.accesses_per_refresh_interval(6.4e9, V0, V_threshold, R_leak, C_cell)
check("Access budget scales linearly with clock frequency (4x clock -> 4x budget)",
      abs(budget_high / budget_low - 4.0) < 1e-9)


clock =  1.6 GHz  ->      113,537,860 address-decode cycles per refresh interval
clock =  3.2 GHz  ->      227,075,720 address-decode cycles per refresh interval
clock =  6.4 GHz  ->      454,151,440 address-decode cycles per refresh interval
PASS  —  Access budget scales linearly with clock frequency (4x clock -> 4x budget)


## Final grade

In [7]:
failures = [label for label, ok in checks if not ok]
print(f"{len(checks) - len(failures)}/{len(checks)} checks passed")

if failures:
    raise AssertionError("Failed checks: " + ", ".join(failures))
else:
    print("\nALL CHECKS PASSED — the Boolean AND-gate address decoder, the arithmetic "
          "row*n_cols+col flat-index formula, and an actual physical DRAM cell's decaying "
          "voltage all agree, exhaustively, across every address tested.")


21/21 checks passed

ALL CHECKS PASSED — the Boolean AND-gate address decoder, the arithmetic row*n_cols+col flat-index formula, and an actual physical DRAM cell's decaying voltage all agree, exhaustively, across every address tested.
